# 06 - Global SHAP Analysis

**Dissertation:** Explainable Machine Learning for Locational Frequency Stability Assessment (IEEE 9-bus)

This notebook loads pre-computed SHAP values and produces publication-quality global interpretability figures: feature importance tables, cross-bus heatmaps, bar charts, beeswarm plots, and ranking consistency analysis (methodology: Kilembe, Hamilton & Papadopoulos 2025, IJEPES 170).

## Cell 1 - Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pickle

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from scipy import stats

BASE_PATH = '/content/drive/MyDrive/Dissertation_ML_Project_V1'
os.chdir(BASE_PATH)

shap_dir = os.path.join(BASE_PATH, 'shap_results')
processed_dir = os.path.join(BASE_PATH, 'data', 'processed')

with open(os.path.join(shap_dir, 'all_shap_values.pkl'), 'rb') as f:
    all_shap = pickle.load(f)

missing_algorithms = []

rf_shap_path = os.path.join(shap_dir, 'rf', 'all_shap_values_rf.pkl')
if os.path.exists(rf_shap_path):
    with open(rf_shap_path, 'rb') as f:
        rf_shap = pickle.load(f)
    print(f'Loaded RF SHAP: {len(rf_shap)} targets')
else:
    missing_algorithms.append('RF')

xgb_shap_path = os.path.join(shap_dir, 'xgboost', 'all_shap_values_xgb.pkl')
if os.path.exists(xgb_shap_path):
    with open(xgb_shap_path, 'rb') as f:
        xgb_shap = pickle.load(f)
    print(f'Loaded XGBoost SHAP: {len(xgb_shap)} targets')
else:
    missing_algorithms.append('XGBoost')

if missing_algorithms:
    raise RuntimeError(
        'Missing SHAP results for required algorithm(s): '
        + ', '.join(missing_algorithms)
        + '. Run Notebook 05 successfully before Notebook 06.'
    )

ALL_ALG_SHAP = {'MLP': all_shap, 'RF': rf_shap, 'XGBoost': xgb_shap}

if os.path.exists(os.path.join(shap_dir, 'X_test_raw.pkl')):
    X_test_raw = joblib.load(os.path.join(shap_dir, 'X_test_raw.pkl'))
else:
    X_test_raw = joblib.load(os.path.join(processed_dir, 'X_test_raw.pkl'))

X_test = joblib.load(os.path.join(processed_dir, 'X_test_scaled.pkl'))

FEATURE_NAMES = ['SG 1 MVA', 'SG 2 MVA', 'SG 3 MVA', 'System Loading', 'CIG MW', 'Outage SG', 'SG 1 MW', 'SG 2 MW', 'SG 3 MW']
TARGET_ORDER = (
    [f'RoCoF Bus {i}' for i in range(1, 10)] + ['RoCoF Worst'] +
    [f'Nadir Bus {i}' for i in range(1, 10)] + ['Nadir Worst']
)

fig_out = os.path.join(BASE_PATH, 'figures', 'SHAP')
os.makedirs(fig_out, exist_ok=True)

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9

print(f'Loaded SHAP results for {len(all_shap)} MLP targets.')
print(f'X_test_raw shape: {X_test_raw.shape}')
print(f'Output directory: {fig_out}')


In [ ]:
# Use X_test_raw (unscaled) for feature values in Explanation -> physical units in beeswarm
data_raw = X_test_raw.values if hasattr(X_test_raw, 'values') else np.array(X_test_raw)
if isinstance(X_test_raw, pd.DataFrame) and list(X_test_raw.columns) != FEATURE_NAMES:
    data_raw = X_test_raw[FEATURE_NAMES].values

# Expanded selection: Bus 1 (generator), Bus 2 (vulnerable), Bus 7 (load), Worst (system-wide)
selected = ['RoCoF Bus 3',
            'Nadir Bus 3']

for alg_name, alg_shap_data in ALL_ALG_SHAP.items():
    if not alg_shap_data:
        continue
    for target_name in selected:
        if target_name not in alg_shap_data:
            continue
        ev = alg_shap_data[target_name]['expected_value']
        sv = np.array(alg_shap_data[target_name]['shap_values'])
        explanation = shap.Explanation(
            values=sv,
            base_values=np.full(sv.shape[0], ev),
            data=data_raw,
            feature_names=FEATURE_NAMES
        )
        plt.figure(figsize=(10, 6))
        shap.plots.beeswarm(explanation, show=False)
        plt.title(f'{alg_name} - {target_name}')
        plt.tight_layout()
        safe_name = target_name.replace(' ', '_')
        plt.savefig(os.path.join(fig_out, f'beeswarm_{alg_name}_{safe_name}.png'), dpi=300, bbox_inches='tight')
        plt.show()

    # Combined 2x4 beeswarm per algorithm (first 8 selected)
    fig, axes = plt.subplots(2, 4, figsize=(18, 10))
    axes = axes.flatten()
    for idx, target_name in enumerate(selected[:8]):
        if target_name not in alg_shap_data:
            continue
        ev = alg_shap_data[target_name]['expected_value']
        sv = np.array(alg_shap_data[target_name]['shap_values'])
        explanation = shap.Explanation(values=sv, base_values=np.full(sv.shape[0], ev), data=data_raw, feature_names=FEATURE_NAMES)
        shap.plots.beeswarm(explanation, show=False, ax=axes[idx], plot_size=None)
        axes[idx].set_title(target_name)
    plt.suptitle(f'{alg_name} - SHAP beeswarm (unscaled feature values)', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_out, f'beeswarm_combined_{alg_name}.png'), dpi=300, bbox_inches='tight')
    plt.show()

print('--- Analytical commentary: Beeswarm uses unscaled feature values for colour; red = high, blue = low.')

## Cell 2 - Compute Global Feature Importance Table

In [ ]:
# Mean |SHAP| per feature for each algorithm -> dict of DataFrames
ALL_IMPORTANCE = {}
for alg_name, alg_data in ALL_ALG_SHAP.items():
    if not alg_data:
        continue
    mean_abs = {}
    for target_name, data in alg_data.items():
        sv = np.array(data['shap_values'])
        mean_abs[target_name] = np.abs(sv).mean(axis=0)
    imp_df = pd.DataFrame(mean_abs, index=FEATURE_NAMES)
    order_cols = [t for t in TARGET_ORDER if t in imp_df.columns]
    imp_df = imp_df[order_cols]
    ALL_IMPORTANCE[alg_name] = imp_df
    imp_df.to_csv(os.path.join(fig_out, f'global_feature_importance_{alg_name}.csv'))

# Backward compat: importance_df = MLP for cells that reference it
importance_df = ALL_IMPORTANCE.get('MLP', pd.DataFrame())
rank_df = importance_df.rank(ascending=False, method='min').astype(int) if not importance_df.empty else pd.DataFrame()

for alg_name, imp_df in ALL_IMPORTANCE.items():
    print(f'\n=== {alg_name} Mean |SHAP| per feature ===')
    display(imp_df.round(4))
    print(f'{alg_name} Ranks (1 = most important):')
    display(imp_df.rank(ascending=False, method='min').astype(int))

## Cell 3 - Cross-Bus SHAP Heatmap: RoCoF

In [ ]:
# Cross-bus heatmap per algorithm (methodology: "produced separately for each algorithm")
rocof_targets = [f'RoCoF Bus {i}' for i in range(1, 10)] + ['RoCoF Worst']
rocof_labels = ['Bus 1', 'Bus 2', 'Bus 3', 'Bus 4', 'Bus 5', 'Bus 6', 'Bus 7', 'Bus 8', 'Bus 9', 'Worst']

for alg_name in ALL_IMPORTANCE:
    imp_df = ALL_IMPORTANCE[alg_name]
    available_rocof = [t for t in rocof_targets if t in imp_df.columns]
    if not available_rocof:
        continue
    M_rocof = imp_df[available_rocof].copy()
    M_rocof.columns = [t.replace('RoCoF ', '') for t in available_rocof]

    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(M_rocof, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Mean |SHAP| (Hz/s)'})
    ax.set_title(f'{alg_name} - Mean |SHAP| RoCoF Models (Hz/s)')
    ax.set_xlabel('Bus / System-wide')
    ax.set_ylabel('Feature')
    ax.tick_params(axis='both', which='major', labelsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_out, f'heatmap_RoCoF_{alg_name}.png'), dpi=300, bbox_inches='tight')
    plt.show()

    M_rocof_norm = M_rocof / M_rocof.sum(axis=0)
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(M_rocof_norm, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Normalized (column sum=1)'})
    ax.set_title(f'{alg_name} - Normalized Mean |SHAP| RoCoF')
    ax.set_xlabel('Bus / System-wide')
    ax.set_ylabel('Feature')
    plt.tight_layout()
    plt.savefig(os.path.join(fig_out, f'heatmap_RoCoF_normalized_{alg_name}.png'), dpi=300, bbox_inches='tight')
    plt.show()

print('--- Analytical commentary: Absolute mean |SHAP| shows which features drive RoCoF at each bus; normalized view shows relative importance per bus.')

## Cell 4 - Cross-Bus SHAP Heatmap: Nadir

In [ ]:
nadir_targets = [f'Nadir Bus {i}' for i in range(1, 10)] + ['Nadir Worst']

for alg_name in ALL_IMPORTANCE:
    imp_df = ALL_IMPORTANCE[alg_name]
    available_nadir = [t for t in nadir_targets if t in imp_df.columns]
    if not available_nadir:
        continue
    M_nadir = imp_df[available_nadir].copy()
    M_nadir.columns = [t.replace('Nadir ', '') for t in available_nadir]

    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(M_nadir, annot=True, fmt='.4f', cmap='Blues', ax=ax, cbar_kws={'label': 'Mean |SHAP| (Hz)'})
    ax.set_title(f'{alg_name} - Mean |SHAP| Nadir Models (Hz)')
    ax.set_xlabel('Bus / System-wide')
    ax.set_ylabel('Feature')
    ax.tick_params(axis='both', which='major', labelsize=9)
    plt.tight_layout()
    os.makedirs(fig_out, exist_ok=True) # Ensure directory exists before saving
    plt.savefig(os.path.join(fig_out, f'heatmap_Nadir_{alg_name}.png'), dpi=300, bbox_inches='tight')
    plt.show()

    M_nadir_norm = M_nadir / M_nadir.sum(axis=0)
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(M_nadir_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax, cbar_kws={'label': 'Normalized (column sum=1)'})
    ax.set_title(f'{alg_name} - Normalized Mean |SHAP| Nadir')
    ax.set_xlabel('Bus / System-wide')
    ax.set_ylabel('Feature')
    plt.tight_layout()
    os.makedirs(fig_out, exist_ok=True) # Ensure directory exists before saving
    plt.savefig(os.path.join(fig_out, f'heatmap_Nadir_normalized_{alg_name}.png'), dpi=300, bbox_inches='tight')
    plt.show()

print('--- Analytical commentary: Nadir reflects governor response; importance patterns differ from RoCoF across buses.')

## Cell 5 - Combined RoCoF vs Nadir Heatmap (Side by Side)

In [ ]:
# Bus 1-9 only (no Worst) for direct comparison
bus_only = [f'Bus {i}' for i in range(1, 10)]
M_rocof_bus = importance_df[[f'RoCoF Bus {i}' for i in range(1, 10)]].copy()
M_rocof_bus.columns = bus_only
M_nadir_bus = importance_df[[f'Nadir Bus {i}' for i in range(1, 10)]].copy()
M_nadir_bus.columns = bus_only

vmin = min(M_rocof_bus.min().min(), M_nadir_bus.min().min())
vmax = max(M_rocof_bus.max().max(), M_nadir_bus.max().max())

fig, axes = plt.subplots(1, 2, figsize=(12, 8))

sns.heatmap(M_rocof_bus, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0], vmin=vmin, vmax=vmax,
            cbar_kws={'label': 'Mean |SHAP|'})
axes[0].set_title('RoCoF (Hz/s)')
axes[0].set_xlabel('Bus')
axes[0].set_ylabel('Feature')

sns.heatmap(M_nadir_bus, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[1], vmin=vmin, vmax=vmax,
            cbar_kws={'label': 'Mean |SHAP|'})
axes[1].set_title('Nadir (Hz)')
axes[1].set_xlabel('Bus')
axes[1].set_ylabel('Feature')

plt.suptitle('MLP - Mean |SHAP| - RoCoF vs Nadir by Bus (same scale)', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(fig_out, 'heatmap_combined_RoCoF_vs_Nadir_MLP.png'), dpi=300, bbox_inches='tight')
plt.show()

print('\n--- Analytical commentary ---')
print('Same colormap and scale allow direct comparison: different features drive RoCoF (initial rate of change) vs Nadir (minimum frequency). Locational variation is visible across buses.')

## Cell 6 - Global Feature Importance Bar Charts (Selected Models)

In [ ]:
selected = ['RoCoF Bus 2', 'RoCoF Worst', 'Nadir Bus 2', 'Nadir Worst']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, target_name in enumerate(selected):
    ax = axes[idx]
    vals = importance_df[target_name].sort_values(ascending=True)
    colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(vals)))
    ax.barh(vals.index, vals.values, color=colors)
    ax.set_title(target_name)
    ax.set_xlabel('Mean |SHAP|')
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('MLP - Global feature importance (mean |SHAP|) - selected models', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(fig_out, 'bar_charts_selected_MLP.png'), dpi=300, bbox_inches='tight')
plt.show()

print('\n--- Analytical commentary ---')
print('RoCoF Bus 2 and RoCoF Worst highlight inertia/MVA and power imbalance; Nadir Bus 2 and Nadir Worst show which features drive frequency nadir (often loading, CIG, governor-related).')

## Cell 7 - SHAP Beeswarm/Summary Plots (Selected Models)

In [ ]:
# Use X_test_raw (unscaled) for feature values in Explanation -> physical units in beeswarm
data_raw = X_test_raw.values if hasattr(X_test_raw, 'values') else np.array(X_test_raw)
if isinstance(X_test_raw, pd.DataFrame) and list(X_test_raw.columns) != FEATURE_NAMES:
    data_raw = X_test_raw[FEATURE_NAMES].values

# Expanded selection: Bus 1 (generator), Bus 2 (vulnerable), Bus 7 (load), Worst (system-wide)
selected = ['RoCoF Bus 1', 'RoCoF Bus 2', 'RoCoF Bus 7', 'RoCoF Worst',
            'Nadir Bus 1', 'Nadir Bus 2', 'Nadir Bus 7', 'Nadir Worst']

for alg_name, alg_shap_data in ALL_ALG_SHAP.items():
    if not alg_shap_data:
        continue
    for target_name in selected:
        if target_name not in alg_shap_data:
            continue
        ev = alg_shap_data[target_name]['expected_value']
        sv = np.array(alg_shap_data[target_name]['shap_values'])
        explanation = shap.Explanation(
            values=sv,
            base_values=np.full(sv.shape[0], ev),
            data=data_raw,
            feature_names=FEATURE_NAMES
        )
        plt.figure(figsize=(10, 6))
        shap.plots.beeswarm(explanation, show=False)
        plt.title(f'{alg_name} - {target_name}')
        plt.tight_layout()
        safe_name = target_name.replace(' ', '_')
        plt.savefig(os.path.join(fig_out, f'beeswarm_{alg_name}_{safe_name}.png'), dpi=300, bbox_inches='tight')
        plt.show()

    # Combined 2x4 beeswarm per algorithm (first 8 selected)
    fig, axes = plt.subplots(2, 4, figsize=(18, 10))
    axes = axes.flatten()
    for idx, target_name in enumerate(selected[:8]):
        if target_name not in alg_shap_data:
            continue
        ev = alg_shap_data[target_name]['expected_value']
        sv = np.array(alg_shap_data[target_name]['shap_values'])
        explanation = shap.Explanation(values=sv, base_values=np.full(sv.shape[0], ev), data=data_raw, feature_names=FEATURE_NAMES)
        shap.plots.beeswarm(explanation, show=False, ax=axes[idx], plot_size=None)
        axes[idx].set_title(target_name)
    plt.suptitle(f'{alg_name} - SHAP beeswarm (unscaled feature values)', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_out, f'beeswarm_combined_{alg_name}.png'), dpi=300, bbox_inches='tight')
    plt.show()

print('--- Analytical commentary: Beeswarm uses unscaled feature values for colour; red = high, blue = low.')

## Cell 8 - Top-3 Features per Bus Summary Table

In [ ]:
for alg_name, imp_df in ALL_IMPORTANCE.items():
    rows = []
    for bus in range(1, 10):
        for metric, label in [('RoCoF', 'RoCoF'), ('Nadir', 'Nadir')]:
            target = f'{metric} Bus {bus}'
            if target not in imp_df.columns:
                continue
            top3 = imp_df[target].nlargest(3)
            rows.append({
                'Bus': bus,
                'Metric': label,
                'Rank 1': top3.index[0],
                'Rank 2': top3.index[1],
                'Rank 3': top3.index[2],
            })
    top3_df = pd.DataFrame(rows)
    print(f'\n=== {alg_name} - Top-3 Features per Bus ===')
    display(top3_df)
    out_csv = os.path.join(fig_out, f'top3_features_per_bus_{alg_name}.csv')
    top3_df.to_csv(out_csv, index=False)
    print(f'Saved: {out_csv}')

## Cell 9 - Feature Importance Ranking Consistency (Spearman Correlation)

In [ ]:
bus_cols_rocof = [f'RoCoF Bus {i}' for i in range(1, 10)]
bus_cols_nadir = [f'Nadir Bus {i}' for i in range(1, 10)]

for alg_name, imp_df in ALL_IMPORTANCE.items():
    available_rocof = [c for c in bus_cols_rocof if c in imp_df.columns]
    available_nadir = [c for c in bus_cols_nadir if c in imp_df.columns]

    if available_rocof:
        rank_rocof = imp_df[available_rocof].rank(ascending=False, method='min').astype(int)
        corr_rocof = rank_rocof.corr(method='spearman')
        labels_r = [c.replace('RoCoF ', '') for c in available_rocof]
        corr_rocof.index = corr_rocof.columns = labels_r
        plt.figure(figsize=(6, 5))
        sns.heatmap(corr_rocof, annot=True, fmt='.2f', cmap='RdYlGn', center=0.5, vmin=0, vmax=1,
                    cbar_kws={'label': 'Spearman correlation'})
        plt.title(f'{alg_name} - RoCoF: feature-rank correlation across buses')
        plt.xlabel('Bus')
        plt.ylabel('Bus')
        plt.tight_layout()
        plt.savefig(os.path.join(fig_out, f'ranking_correlation_RoCoF_{alg_name}.png'), dpi=300, bbox_inches='tight')
        plt.show()

    if available_nadir:
        rank_nadir = imp_df[available_nadir].rank(ascending=False, method='min').astype(int)
        corr_nadir = rank_nadir.corr(method='spearman')
        labels_n = [c.replace('Nadir ', '') for c in available_nadir]
        corr_nadir.index = corr_nadir.columns = labels_n
        plt.figure(figsize=(6, 5))
        sns.heatmap(corr_nadir, annot=True, fmt='.2f', cmap='RdYlGn', center=0.5, vmin=0, vmax=1,
                    cbar_kws={'label': 'Spearman correlation'})
        plt.title(f'{alg_name} - Nadir: feature-rank correlation across buses')
        plt.xlabel('Bus')
        plt.ylabel('Bus')
        plt.tight_layout()
        plt.savefig(os.path.join(fig_out, f'ranking_correlation_Nadir_{alg_name}.png'), dpi=300, bbox_inches='tight')
        plt.show()

print('\n--- Spearman correlation of feature importance rankings across buses, per algorithm.')
print('High values = same features matter at different buses; low values support locational thesis.')

## Cell 10 - Cross-Algorithm SHAP Consistency (Key Original Contribution)

For each target, compare feature importance rankings (mean |SHAP|) across MLP, RF, and XGBoost. Spearman ρ measures agreement; high ρ = algorithms agree on which features matter.

In [ ]:
from scipy.stats import spearmanr

cross_alg_rows = []
for target_name in TARGET_ORDER:
    rankings = {}
    for alg_name, alg_data in ALL_ALG_SHAP.items():
        if target_name not in alg_data:
            continue
        sv = np.array(alg_data[target_name]['shap_values'])
        importance = np.abs(sv).mean(axis=0)
        rankings[alg_name] = importance

    missing_for_target = [alg for alg in ['MLP', 'RF', 'XGBoost'] if alg not in rankings]
    if missing_for_target:
        raise RuntimeError(
            f'Missing cross-algorithm SHAP data for target {target_name}: '
            + ', '.join(missing_for_target)
        )

    pairs = [('MLP', 'RF'), ('MLP', 'XGBoost'), ('RF', 'XGBoost')]
    row = {'target': target_name}
    rhos = []
    for a, b in pairs:
        if a in rankings and b in rankings:
            rho, pval = spearmanr(rankings[a], rankings[b])
            row[f'{a}_vs_{b}_rho'] = rho
            row[f'{a}_vs_{b}_pval'] = pval
            rhos.append(rho)
        else:
            row[f'{a}_vs_{b}_rho'] = np.nan
            row[f'{a}_vs_{b}_pval'] = np.nan
    row['mean_rho'] = np.nanmean(rhos) if rhos else np.nan
    row['consistent'] = row['mean_rho'] >= 0.7 if not np.isnan(row['mean_rho']) else False
    cross_alg_rows.append(row)

cross_alg_df = pd.DataFrame(cross_alg_rows)
display(cross_alg_df.round(3))

cross_alg_df.to_csv(os.path.join(fig_out, 'cross_algorithm_consistency.csv'), index=False)
print(f'\nMean cross-algorithm Spearman rho: {cross_alg_df["mean_rho"].mean():.3f}')
print(f'Targets with mean rho >= 0.7 (consistent): {cross_alg_df["consistent"].sum()} / {len(cross_alg_df)}')
print(f'Targets with mean rho < 0.7 (algorithm-dependent): {(~cross_alg_df["consistent"]).sum()}')

# Heatmap of cross-algorithm consistency
rho_cols = [c for c in cross_alg_df.columns if c.endswith('_rho') and c != 'mean_rho']
if rho_cols:
    plot_df = cross_alg_df.set_index('target')[rho_cols + ['mean_rho']].copy()
    plot_df.columns = [c.replace('_rho', '').replace('_vs_', ' vs ') for c in rho_cols] + ['Mean']
    fig, ax = plt.subplots(figsize=(10, 14))
    sns.heatmap(plot_df, annot=True, fmt='.2f', cmap='RdYlGn', center=0.7, vmin=0, vmax=1, ax=ax, cbar_kws={'label': 'Spearman ρ'})
    ax.set_title('Cross-Algorithm SHAP Ranking Consistency')
    ax.set_xlabel('Algorithm Pair')
    ax.set_ylabel('Target')
    plt.tight_layout()
    plt.savefig(os.path.join(fig_out, 'cross_algorithm_consistency_heatmap.png'), dpi=300, bbox_inches='tight')
    plt.show()